# `test_core.py` Walkthrough

## 1. Overview

**Core Question:** How do we prevent COVID-19 type pathogen outbreaks in a correctional setting using activity schedule and movement mapping as tools for public health planning?

This notebook walks through the 17 tests in `tests/test_core.py`, grouped into two threads:

1. **Activity schedules**: data loading &rarr; schedule mechanism &rarr; place selection
2. **Movement mapping**: place counting &rarr; co-location &rarr; transmission

The `core` tests were written by Nick Collier (`@ncollier`) (ANL) and fall into four groups:
- Creation tests (n=3)
- Selection tests (n=3)
- Matrix assignment tests (n=2)
- Disease tests (n=9)

All tests use a small synthetic facility defined in `test_data/`.

In [6]:
import os
os.environ["UCX_TLS"] = "tcp"
os.environ["OMPI_MCA_opal_warn_on_missing_libcuda"] = "0"
os.environ["OMPI_MCA_btl"] = "tcp,self"
os.chdir("/oscar/home/akhann16/code/radmodel")

import numpy as np
import pandas as pd
import yaml
import tempfile
from mpi4py import MPI

from radmodel import population, common, core

## 2. Test Data

The tests use `ng_*` ("next-gen") CSV files in `test_data/`. 
These replaced an older set of files without the `ng_` prefix. The older set is no longer referenced anywhere.

Key differences from the old files:
- Residents have **distinct activity columns** (`morning_act`, `noon_act`, `evening_act`) 
- 1200 residents, 504 places, 1 schedule; matching the medium facility scale
- The old files had 20 residents, 113 places, and 10 schedule types

Currently, there is only one schedule (schedule `0`) assigned to everyone.

In [7]:
# ng_schedules.csv - the daily schedule template
schedules_df = pd.read_csv("test_data/ng_schedules.csv")
schedules_df

,schedule_id,start,place_type,risk,time
0,0,0,cell,1.0,0
1,0,435,cafeteria,1.0,715
2,0,510,morning_act,1.0,830
3,0,630,cell,1.0,1030
4,0,675,cafeteria,1.0,1115
5,0,750,noon_act,2.0,1230
6,0,930,cell,1.0,1530
7,0,960,cafeteria,1.0,1600
8,0,1080,evening_act,1.5,1800
9,0,1230,cell,1.0,2030


### 2.1 Schedules (`ng_schedules.csv`)

This schedule implies that there are 9 key timepoints during the course of the day. 
    
- The `schedule_id` column shows there is only one schedule id currently.
- The zeroth row shows that start time shows the time in minutes since 0000 HHMM (i.e., midnight). Time 435 = 0715 in HHMM (rightmost column `time`). 
- The $i$th row in `place_type` column shows where an agent following the assigned schedule enters (if they are not already there) at time HHMM in row $i$.
- The agent stays in the place corresponding to the $i$th row until the start of the activity at time $i+1.$
- The `risk` column is a multiplier on transmission probability (e.g., noon activity = 2.0, evening activity = 1.5). These values are hypothetical and have not been estimated or derived from data.
- The `time` column converts the `start` (measured in minutes since 0000) to a human readable time in HHMM.

In [30]:
# ng_places.csv - 504 places: 500 cells + cafeteria + gym + yard + classroom
places_df = pd.read_csv("test_data/ng_places.csv")
print(f"Total places: {len(places_df)}")
print(f"\nFirst 5:")
display(places_df.head())
print(f"\nLast 6:")
display(places_df.tail(6))

Total places: 504

First 5:


,place_id,name,type
0,0,cell_0,cell
1,1,cell_1,cell
2,2,cell_2,cell
3,3,cell_3,cell
4,4,cell_4,cell



Last 6:


,place_id,name,type
498,498,cell_498,cell
499,499,cell_499,cell
500,500,cafeteria_1,cafeteria
501,501,gym_1,gym
502,502,yard_1,yard
503,503,classroom_1,education


### 2.2 Places (`ng_places.csv`)

The places dataframe shows the existing place structure in the facility.

- The `place_id` column shows 500 cells (numbered 0...499). 
- Other places include one cafeteria, gym, yard, and classroom (as shown in the name column).
- The `type` column is currently redundant but will become meaningful when more columns are added.

In [25]:
# ng_residents.csv - 1200 residents, each assigned to a cell + shared places
residents_df = pd.read_csv("test_data/ng_residents.csv")
print(f"Total residents: {len(residents_df)}")
print(f"\nFirst 6:")
display(residents_df.head(6))
print(f"\nLast 6:")
display(residents_df.tail(6))

cell_counts = residents_df["cell"].value_counts()
print(f"Total unique cells: {len(cell_counts)}")
print(f"Cells with 2 residents: {(cell_counts == 2).sum()}")
print(f"Cells with 3 residents: {(cell_counts == 3).sum()}")

Total residents: 1200

First 6:


,person_id,schedule_id,cell,cafeteria,morning_act,noon_act,evening_act,mod
0,0,0,0,500,501,503,502,0
1,1,0,1,500,503,501,502,1
2,2,0,2,500,502,503,501,2
3,3,0,3,500,501,502,503,3
4,4,0,4,500,502,501,503,4
5,5,0,5,500,503,502,501,5



Last 6:


,person_id,schedule_id,cell,cafeteria,morning_act,noon_act,evening_act,mod
1194,1194,0,194,500,501,503,502,0
1195,1195,0,195,500,503,501,502,1
1196,1196,0,196,500,502,503,501,2
1197,1197,0,197,500,501,502,503,3
1198,1198,0,198,500,502,501,503,4
1199,1199,0,199,500,503,502,501,5


Total unique cells: 500
Cells with 2 residents: 300
Cells with 3 residents: 200


### 2.3 Residents (`ng_residents.csv`)

The `residents.csv` dataset shows information for each resident.

- The resident's ID in `person_id`.
- The schedule ID they are assigned (currently 0 for everyone).
- Their home cell number in `cell`.
- Their cafeteria number (currently 500 for everyone, since there is only one cafeteria). 
- Where the person goes for their morning, noon, and evening activities (`morning_act`, `noon_act`, `evening_act`) per schedule 0.
- The `mod` column controls which activity places are assigned to each resident (cycles 0&ndash;5), producing variation in where people go even though they share the same schedule.

### 2.4 Putting it all together

There are two underlying look up tables for individual movement:  
- residents.csv: Tells us everything about person X: their schedule number, and their specific place IDs for each activity (cell=17, cafeteria=500, morning_act=503, noon_act=502, evening_act=501) and their mod assignment. 
- schedules.csv: Tells us what type of activity is happening at time T (e.g., at 07:15 everyone on schedule 0 should be at their cafeteria).
- Combine: schedule says "cafeteria time" → look at person X's cafeteria column → place 500. Done.


In short:
- Schedule: "what type of activity at time Y?"  →  column name (e.g., cafeteria)
- Residents.csv  →  "what's person X's place for that activity?"  →  place ID Z (e.g., 500)
- (X, Y, Z) defines person X, time Y, location ID Z.

  
There is an additional `places.csv` dataset that allows us to assess properties of each location.
- What type of place is Z (cell, cafeteria, gym etc.).

### 2.5 How the data were generated

- Mod assignment controls the type of morning, noon, and evening activity for each resident, 
per the [module definition](https://github.com/khanna-lab/radmodel/blob/live-coding-prep/test_data/module_definition.yaml).
- For example, mod 0 means: morning=gym, noon=education, evening=yard.
- Mod is applied during **data generation** (`genpop/generate.py`): for each person, the generator looks up their mod's activity types, picks a specific place of each type  from `places.csv`, and writes the resulting place IDs into the `morning_act`, `noon_act`,  `evening_act` columns of `residents.csv`.
- By the time the model runs, mod has already been resolved. The model only reads  the place ID columns, not the mod column itself.

## 3. Tests

### 3.1 Creation Tests (n=3)

```
pytest tests/test_core.py -k "create" -v
```

These test the data loading pipeline: CSV files &rarr; numpy arrays. The functions live in `src/radmodel/population.py` as module-level functions (not tied to a class):
- `create_schedules(fname)`
- `create_places(fname)` 
- `create_residents(fname, place_id_map, schedule_id_map)`

### `test_create_schedule`

The schedule parsing pipeline converts human-readable time blocks into the model's internal tick-based lookup table:

1. **CSV** &rarr; human-readable blocks (`cafeteria` starts at minute 435 = 07:15)
2. **`ScheduleRow`** &rarr; intermediate representation with start, end, place_type, risk
3. **96-element array** &rarr; model's internal clock (`schedules[29] = P_CAF_IDX`)

### Interpretation
- The schedule array stores **column indices into the resident array**. Not *place IDs* themselves.
- So `schedules[29] = P_CAF_IDX = 4` means "at tick 29, look at column 4 of the resident row to find which cafeteria this person goes to."
- This pointer-based direction scheme is how 1200 people share one schedule template but go to their own assigned places.

In [26]:
# Run create_schedules and inspect the output
id_map, schedules, risks = population.create_schedules("test_data/ng_schedules.csv")

print(f"Schedule ID map: {id_map}")
print(f"Schedules array shape: {schedules.shape}  (= {common.TICKS_PER_DAY} ticks/day x {len(id_map)} schedule)")
print(f"Risks array shape:     {risks.shape}")

Schedule ID map: {0: 0}
Schedules array shape: (96,)  (= 96 ticks/day x 1 schedule)
Risks array shape:     (96,)


In [27]:
# The column index constants and what they map to
print("SCHEDULE_PLACE_TYPE_MAP (place_type string -> resident array column index):")
for name, idx in population.SCHEDULE_PLACE_TYPE_MAP.items():
    print(f"  {name:>12s} -> column {idx}")

print(f"\nResident array columns:")
print(f"  col 0: P_ID_IDX           (person id)")
print(f"  col 1: P_SCHEDULE_IDX     (which schedule)")
print(f"  col 2: P_CURRENT_PLACE_IDX (where they are now)")
print(f"  col 3: P_CELL_IDX         (their cell)")
print(f"  col 4: P_CAF_IDX          (their cafeteria)")
print(f"  col 5: P_MACT_IDX         (morning activity place)")
print(f"  col 6: P_NACT_IDX         (noon activity place)")
print(f"  col 7: P_EACT_IDX         (evening activity place)")
print(f"  col 8: P_STATE_IDX        (disease state)")
print(f"  col 9: P_NEXT_STATE_T_IDX (tick of next state change)")

SCHEDULE_PLACE_TYPE_MAP (place_type string -> resident array column index):
          cell -> column 3
      noon_act -> column 6
   morning_act -> column 5
   evening_act -> column 7
     cafeteria -> column 4

Resident array columns:
  col 0: P_ID_IDX           (person id)
  col 1: P_SCHEDULE_IDX     (which schedule)
  col 2: P_CURRENT_PLACE_IDX (where they are now)
  col 3: P_CELL_IDX         (their cell)
  col 4: P_CAF_IDX          (their cafeteria)
  col 5: P_MACT_IDX         (morning activity place)
  col 6: P_NACT_IDX         (noon activity place)
  col 7: P_EACT_IDX         (evening activity place)
  col 8: P_STATE_IDX        (disease state)
  col 9: P_NEXT_STATE_T_IDX (tick of next state change)


In [ ]:
# The full schedule as a readable tick table
# This is what test_create_schedule verifies against a hand-computed expected array
idx_to_name = {v: k for k, v in population.SCHEDULE_PLACE_TYPE_MAP.items()}

tick_data = []
for i in range(len(schedules)):
    hours = (i * 15) // 60
    mins = (i * 15) % 60
    tick_data.append({
        "tick": i,
        "time": f"{hours:02d}:{mins:02d}",
        "place_type": idx_to_name.get(schedules[i], str(schedules[i])),
        "col_idx": int(schedules[i]),
        "risk": float(risks[i])
    })

tick_df = pd.DataFrame(tick_data)
tick_df

### `test_create_places` and `test_create_residents`

In [28]:
# create_places: CSV -> Places object with place_data array and place_id_map
places = population.create_places("test_data/ng_places.csv")

print(f"place_data shape: {places.place_data.shape}  (504 places x 3 columns: [place_id, person_count, infected_count])")
print(f"place_id_map: identity map {{0:0, 1:1, ..., 503:503}}? {places.place_id_map == {i: i for i in range(504)}}")

place_data shape: (504, 3)  (504 places x 3 columns: [place_id, person_count, infected_count])
place_id_map: identity map {0:0, 1:1, ..., 503:503}? True


In [29]:
# create_residents: CSV -> numpy array (1200 x 10)
schedule_id_map, schedule_data, risks = population.create_schedules("test_data/ng_schedules.csv")
residents = population.create_residents("test_data/ng_residents.csv", places.place_id_map, schedule_id_map)

print(f"Residents shape: {residents.shape}  ({residents.shape[0]} people x {residents.shape[1]} columns)")
print(f"\nResident 17 (spot check):")
print(f"  ID:              {residents[17, population.P_ID_IDX]}")
print(f"  Schedule:        {residents[17, population.P_SCHEDULE_IDX]}")
print(f"  Current place:   {residents[17, population.P_CURRENT_PLACE_IDX]}")
print(f"  Cell:            {residents[17, population.P_CELL_IDX]}")
print(f"  Cafeteria:       {residents[17, population.P_CAF_IDX]}")
print(f"  Morning act:     {residents[17, population.P_MACT_IDX]}  (classroom=503)")
print(f"  Noon act:        {residents[17, population.P_NACT_IDX]}  (yard=502)")
print(f"  Evening act:     {residents[17, population.P_EACT_IDX]}  (gym=501)")
print(f"  Disease state:   {residents[17, population.P_STATE_IDX]}  (SUSCEPTIBLE={common.SUSCEPTIBLE})")
print(f"  Next state tick: {residents[17, population.P_NEXT_STATE_T_IDX]}  (max uint32 = no transition scheduled)")

Residents shape: (1200, 10)  (1200 people x 10 columns)

Resident 17 (spot check):
  ID:              17
  Schedule:        0
  Current place:   17
  Cell:            17
  Cafeteria:       500
  Morning act:     503  (classroom=503)
  Noon act:        502  (yard=502)
  Evening act:     501  (gym=501)
  Disease state:   0  (SUSCEPTIBLE=0)
  Next state tick: 4294967295  (max uint32 = no transition scheduled)


### 3.2 Selection Tests (n=3)

```
pytest tests/test_core.py -k "select" -v
```

This is the heart of the behavioral model. At each tick, `select_next_place(tick)` looks up the schedule to determine which **column** of the resident array to read, then reads that column to get the **place ID**.

For example, at tick 29 (07:15, cafeteria time):
- `schedules[29]` = `P_CAF_IDX` = 4, meaning "look at column 4"
- `residents[17, 4]` = 500, meaning "resident 17 goes to place 500 (the cafeteria)"
- `residents[0, 4]` = 500 too &mdash; everyone shares the same cafeteria

But at tick 0 (midnight, cell time):
- `schedules[0]` = `P_CELL_IDX` = 3, meaning "look at column 3"
- `residents[17, 3]` = 17, meaning "resident 17 goes to cell 17"
- `residents[0, 3]` = 0 &mdash; each person goes to their own cell

In [ ]:
# Demonstrate the column-pointer indirection for a few residents at different ticks
def _init_data():
    """Same helper used by test_core.py"""
    schedule_id_map, schedule_data, risks = population.create_schedules("test_data/ng_schedules.csv")
    places = population.create_places("test_data/ng_places.csv")
    residents = population.create_residents("test_data/ng_residents.csv", places.place_id_map, schedule_id_map)
    with open("test_data/params.yaml") as fin:
        params = yaml.safe_load(fin)
    params["counts_log_file"] = os.path.join(tempfile.gettempdir(), "counts.csv")
    params["places_log_file"] = os.path.join(tempfile.gettempdir(), "place_counts.csv")
    return schedule_data, risks, places, residents, params

schedule_data, risks, places, residents, params = _init_data()

# Trace the indirection manually for a few ticks and residents
print(f"{'tick':>4}  {'time':>5}  {'sched says':>12}  {'col':>3}  {'res 0 place':>11}  {'res 17 place':>12}")
print("-" * 58)
for tick in [0, 29, 34, 50, 72]:
    hours = (tick * 15) // 60
    mins = (tick * 15) % 60
    col = schedule_data[tick]
    place_type = idx_to_name.get(col, str(col))
    r0_place = residents[0, col]
    r17_place = residents[17, col]
    print(f"{tick:4d}  {hours:02d}:{mins:02d}  {place_type:>12}  {col:3d}  {r0_place:>11}  {r17_place:>12}")

### `test_select_next_place`

Runs 144 ticks (1.5 days) and verifies that every resident's `P_CURRENT_PLACE_IDX` matches the place the schedule points to.

### `test_select_next_place_count` and `test_select_next_place_inf_count`

Movement has a second consequence: the model tracks how many people (and how many **infectious** people) are at each place per tick. These tests verify the bookkeeping stays consistent as everyone moves around. This is the bridge from movement to transmission.

In [ ]:
# Run select_next_place and watch the counts change
schedule_data, _, places, residents, params = _init_data()
duration_matrix = core.create_duration_matrix(params)
trans_matrix = core.create_trans_matrix(params["transition_matrix"])
model = core.Model(MPI.COMM_WORLD, schedule_data, residents, places, 0.0,
                   trans_matrix, duration_matrix, params["random_seed"], params)

# Show where everyone is at a few key ticks
for tick in [0, 29, 34, 50]:
    model.select_next_place(tick)
    hours = (tick * 15) // 60
    mins = (tick * 15) % 60
    place_type = idx_to_name.get(schedule_data[tick], "?")
    
    # Count people per place
    counts = {}
    for x in range(residents.shape[0]):
        pid = residents[x, population.P_CURRENT_PLACE_IDX]
        counts[pid] = counts.get(pid, 0) + 1
    
    n_occupied = len(counts)
    biggest = max(counts.values())
    biggest_place = max(counts, key=counts.get)
    
    print(f"Tick {tick:3d} ({hours:02d}:{mins:02d}, {place_type:>12}): "
          f"{n_occupied} places occupied, largest = place {biggest_place} with {biggest} people")

### 3.3 Matrix Assignment Tests (n=2)

```
pytest tests/test_core.py -k "matrix" -v
```

Two matrices define the disease mechanics:

- **Transition matrix** (8x8): probability of moving from one disease state to another (e.g., E&rarr;P: 0.8, E&rarr;I_A: 0.2). Stored as cumulative sums along each row for efficient random sampling.
- **Duration matrix** (8x2): how long someone stays in each state, parameterized as gamma distributions with (k, scale) where scale = mean/k.

In [ ]:
# Disease states for reference
states = ["S", "E", "P", "I_S", "I_A", "R", "H", "D"]

# Transition matrix (raw probabilities from params.yaml, then cumsum'd)
trans_matrix = core.create_trans_matrix(params["transition_matrix"])
print("Transition matrix (cumulative sums for sampling):")
print(f"{'':>4}  " + "  ".join(f"{s:>5}" for s in states))
for i, row in enumerate(trans_matrix):
    print(f"{states[i]:>4}  " + "  ".join(f"{v:5.2f}" for v in row))

print("\nRead each row as: draw uniform [0,1], land in first column where cumsum >= draw")
print("E.g., row E: [0, 0, 0.8, 0, 1.0, ...] means 80% chance -> P, 20% chance -> I_A")

In [ ]:
# Duration matrix: (k, scale) for gamma-distributed sojourn times
duration_matrix = core.create_duration_matrix(params)
print("Duration matrix (gamma distribution parameters in days):")
print(f"{'state':>5}  {'k':>6}  {'scale':>8}  {'mean':>8}")
print("-" * 35)
for i, s in enumerate(states):
    k = duration_matrix[i, 0]
    scale = duration_matrix[i, 1]
    mean = k * scale
    if k > 0:
        print(f"{s:>5}  {k:6.1f}  {scale:8.3f}  {mean:8.1f} days")
    else:
        print(f"{s:>5}  {'n/a':>6}  {'n/a':>8}  {'n/a':>8}")

### 3.4 Disease Tests (n=9)

```
pytest tests/test_core.py -k "disease" -v
```

These split into three subgroups:

#### Transmission boundary tests (n=3)

Test the S&rarr;E transition by controlling `stoe` (susceptible-to-exposed probability):
- `test_disease_update_always_inf` (stoe=1.0): every susceptible co-located with an infectious person becomes EXPOSED
- `test_disease_update_never_inf` (stoe=0.0): nobody becomes EXPOSED
- `test_disease_update_50p_inf` (stoe=0.5): over 1000 trials, count falls in [900, 1100]

In [ ]:
# Demonstrate the stoe boundary: infect resident 0, move everyone to tick 44 (cafeteria),
# then see who gets exposed at stoe=1.0

schedule_data, _, places, residents, params = _init_data()
duration_matrix = core.create_duration_matrix(params)
trans_matrix = core.create_trans_matrix(params["transition_matrix"])

# Infect resident 0
residents[0, population.P_STATE_IDX] = common.INFECTED_SYMP
params["stoe"] = 1.0

model = core.Model(MPI.COMM_WORLD, schedule_data, residents, places, params["stoe"],
                   trans_matrix, duration_matrix, params["random_seed"], params)

# Move everyone to their tick-44 position
tick = 44
model.select_next_place(tick)

# Who is co-located with resident 0?
infected_place = residents[0, population.P_CURRENT_PLACE_IDX]
colocated = [x for x in range(1, residents.shape[0]) if residents[x, population.P_CURRENT_PLACE_IDX] == infected_place]

hours = (tick * 15) // 60
mins = (tick * 15) % 60
place_type = idx_to_name.get(schedule_data[tick], "?")
print(f"Tick {tick} ({hours:02d}:{mins:02d}, {place_type})")
print(f"Resident 0 (INFECTED_SYMP) is at place {infected_place}")
print(f"Co-located susceptible residents: {len(colocated)}  (e.g., {colocated[:5]}...)")

# Now run disease update with stoe=1.0
model.update_disease_state(tick)

n_exposed = np.count_nonzero(residents[1:, population.P_STATE_IDX] == common.EXPOSED)
n_still_susceptible = np.count_nonzero(residents[1:, population.P_STATE_IDX] == common.SUSCEPTIBLE)
print(f"\nAfter update_disease_state (stoe=1.0):")
print(f"  Newly EXPOSED: {n_exposed}  (should == co-located count: {len(colocated)})")
print(f"  Still SUSCEPTIBLE: {n_still_susceptible}")

#### State transition tests (n=2)

- `test_update_disease_states1`: transitions fire at the correct tick, and probabilistic branching (75/25 E&rarr;P vs E&rarr;I_A) produces expected distributions over many trials
- `test_update_disease_states2`: two residents with different `next_state_t` values each transition only at their own scheduled tick

#### Full disease path tests (n=4)

Four end-to-end tests trace a single resident through the complete state machine. Each forces specific transition probabilities to 1.0 to isolate a single path:

| Test | Path | What it covers |
|------|------|---------------|
| `paths1` | S &rarr; E &rarr; P &rarr; I_S &rarr; H &rarr; R &rarr; S | Full hospitalization and recovery |
| `paths2` | S &rarr; E &rarr; I_A &rarr; R &rarr; S | Asymptomatic (skips presymptomatic) |
| `paths3` | S &rarr; E &rarr; P &rarr; I_S &rarr; R &rarr; S | Symptomatic, no hospitalization |
| `paths4` | S &rarr; E &rarr; P &rarr; I_S &rarr; H &rarr; D | Death |

Each test verifies that gamma-distributed sojourn times fall within expected bounds at every stage.

In [ ]:
# Trace disease_state_paths1: S -> E -> P -> I_S -> H -> R -> S
# Force all transitions to follow the hospitalization path
schedule_data, _, place_data, residents, params = _init_data()

residents[0, population.P_STATE_IDX] = common.INFECTED_SYMP
params["stoe"] = 1.0
tm = params["transition_matrix"]
tm["E"]["P"] = 1.0;  tm["E"]["I_A"] = 0
tm["I_S"]["H"] = 1.0; tm["I_S"]["R"] = 0
tm["H"]["R"] = 1.0

model = core.Model(MPI.COMM_WORLD, schedule_data, residents, place_data, params["stoe"],
                   core.create_trans_matrix(tm), core.create_duration_matrix(params),
                   params["random_seed"], params)

# Infect at tick 34 (cafeteria), then follow resident 12
tick = 34
model.select_next_place(tick)
model.update_disease_state(tick)

state_names = {v: k for k, v in common.STATE_MAP.items()}
target = 12

print(f"Tracing resident {target} through the full hospitalization path:")
print(f"{'stage':>12}  {'state':>5}  {'entered_tick':>12}  {'duration_days':>13}")
print("-" * 50)

state = residents[target, population.P_STATE_IDX]
print(f"{'infection':>12}  {state_names[state]:>5}  {tick:>12}  {'':>13}")

while state != common.SUSCEPTIBLE:
    next_t = residents[target, population.P_NEXT_STATE_T_IDX]
    model.update_disease_state(next_t)
    new_state = residents[target, population.P_STATE_IDX]
    duration_days = (next_t - tick) / common.TICKS_PER_DAY
    print(f"{'transition':>12}  {state_names[new_state]:>5}  {next_t:>12}  {duration_days:>13.1f}")
    tick = next_t
    state = new_state

## 4. Summary

The 17 core tests verify the model in layers:

1. **Data loading** (3 tests): CSV &rarr; numpy arrays correctly
2. **Movement** (3 tests): schedule-driven place selection + occupancy counting
3. **Disease parameters** (2 tests): transition probabilities and sojourn time distributions
4. **Disease dynamics** (9 tests): transmission at S&rarr;E boundary, state machine timing, and complete infection trajectories

The first two groups test the **activity schedule and movement mapping** machinery &mdash; the novel intervention levers in the model. The last two groups test the standard **epidemiological state machine** that runs on top of the movement layer.